In [ ]:
import hashlib
import json
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import h5py
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from use_croma import PretrainedCROMA # from https://github.com/antofuller/CROMA


def croma_batch_normalize(
    x: torch.Tensor,
) -> torch.Tensor:
    """
    Batch-wise CROMA normalization.

    For each channel:
        - min = mean - 2 * std
        - max = mean + 2 * std
        - clip to [0, 1]

    Statistics are computed across:
        (batch, height, width)

    Args:
        x: Tensor of shape (B, C, H, W).

    Returns:
        Normalized tensor of shape (B, C, H, W).
    """
    x = x.float()

    imgs = []

    for channel in range(x.shape[1]):
        channel_data = x[:, channel, :, :]

        mean = channel_data.mean()
        std = channel_data.std()

        min_value = mean - 2 * std
        max_value = mean + 2 * std

        denom = max_value - min_value

        if denom == 0:
            denom = torch.tensor(
                1.0,
                device=x.device,
                dtype=x.dtype,
            )

        img = (channel_data - min_value) / denom
        img = torch.clamp(img, 0, 1)

        imgs.append(img.unsqueeze(1))

    return torch.cat(imgs, dim=1)


class So2SatLCZ42Dataset(Dataset):
    """
    So2Sat LCZ42 dataset with optional class filtering and label remapping.

    Features:
        - supports urban-only filtering
        - supports explicit class selection
        - can remap selected labels to a contiguous range [0, C - 1]

    Returned items:
        (sar, optical, label_index)
    """

    def __init__(
        self,
        h5_path: str,
        image_resolution: int = 120,
        urban_only: bool = False,
        selected_classes: Optional[Iterable[int]] = None,
        remap_labels: bool = True,
    ) -> None:
        super().__init__()
        self.h5_path = h5_path
        self.image_resolution = image_resolution
        self.urban_only = urban_only
        self.remap_labels = remap_labels

        self._h5_file = None
        self._sen1 = None
        self._sen2 = None
        self._labels = None

        self.selected_classes = None
        self.label_mapping = None

        if selected_classes is not None:
            self.selected_classes = sorted(set(int(c) for c in selected_classes))
            if self.remap_labels:
                self.label_mapping = {
                    original_label: new_label
                    for new_label, original_label in enumerate(
                        self.selected_classes
                    )
                }

        with h5py.File(self.h5_path, "r") as f:
            labels = f["label"][:]
            class_idx = np.argmax(labels, axis=1)

            mask = np.ones_like(class_idx, dtype=bool)

            if self.urban_only:
                mask &= class_idx < 10

            if self.selected_classes is not None:
                mask &= np.isin(class_idx, self.selected_classes)

            self.indices = np.where(mask)[0]
            self._length = len(self.indices)

    def _lazy_open(self) -> None:
        """
        Open the HDF5 file lazily, once per worker process.
        """
        if self._h5_file is None:
            self._h5_file = h5py.File(self.h5_path, "r")
            self._sen1 = self._h5_file["sen1"]
            self._sen2 = self._h5_file["sen2"]
            self._labels = self._h5_file["label"]

    def __len__(self) -> int:
        return self._length

    @staticmethod
    def _build_sar(sen1: torch.Tensor) -> torch.Tensor:
        """
        Build the 2-channel SAR tensor from sen1.
        """
        sar = sen1[4:6, :, :]
        ## à ajouter sar = 10.0 * torch.log10(torch.clamp(sar, min=1e-8))
        return sar

    @staticmethod
    def _build_optical(sen2: torch.Tensor) -> torch.Tensor:
        """
        Build the 12-channel optical tensor in Sentinel-2 order.
        """
        _, height, width = sen2.shape
        zeros_band = torch.zeros(1, height, width, dtype=sen2.dtype)

        b2 = sen2[0:1, :, :]
        b3 = sen2[1:2, :, :]
        b4 = sen2[2:3, :, :]
        b5 = sen2[3:4, :, :]
        b6 = sen2[4:5, :, :]
        b7 = sen2[5:6, :, :]
        b8a = sen2[6:7, :, :]
        b11 = sen2[7:8, :, :]
        b12 = sen2[8:9, :, :]
        b8 = sen2[9:10, :, :]

        optical = torch.cat(
            [
                zeros_band,
                b2,
                b3,
                b4,
                b5,
                b6,
                b7,
                b8,
                b8a,
                zeros_band,
                b11,
                b12,
            ],
            dim=0,
        )
        return optical

    def __getitem__(
        self,
        index: int,
    ) -> Tuple[torch.Tensor, torch.Tensor, int]:
        self._lazy_open()

        real_index = int(self.indices[index])

        sen1_np = self._sen1[real_index]
        sen2_np = self._sen2[real_index]
        label_one_hot = self._labels[real_index]

        sen1 = torch.from_numpy(sen1_np).permute(2, 0, 1).float()
        sen2 = torch.from_numpy(sen2_np).permute(2, 0, 1).float()

        sar = self._build_sar(sen1)
        optical = self._build_optical(sen2)

        if self.image_resolution is not None:
            sar = F.interpolate(
                sar.unsqueeze(0),
                size=(self.image_resolution, self.image_resolution),
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)

            optical = F.interpolate(
                optical.unsqueeze(0),
                size=(self.image_resolution, self.image_resolution),
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)

        label_index = int(np.argmax(label_one_hot).astype(np.int64))

        if self.label_mapping is not None:
            label_index = self.label_mapping[label_index]

        return sar, optical, label_index

    def close(self) -> None:
        """
        Close the HDF5 file if it is open.
        """
        if self._h5_file is not None:
            self._h5_file.close()
            self._h5_file = None
            self._sen1 = None
            self._sen2 = None
            self._labels = None


def build_cache_path(
    cache_dir: str,
    cache_name: str,
    cache_metadata: Dict,
) -> Path:
    """
    Build a deterministic cache path from metadata.

    Args:
        cache_dir: Root cache directory.
        cache_name: Human-readable cache prefix.
        cache_metadata: Metadata describing the extraction setup.

    Returns:
        Full cache file path.
    """
    cache_dir_path = Path(cache_dir)
    cache_dir_path.mkdir(parents=True, exist_ok=True)

    payload = json.dumps(cache_metadata, sort_keys=True)
    digest = hashlib.md5(payload.encode("utf-8")).hexdigest()

    return cache_dir_path / f"{cache_name}_{digest}.pt"


def reinitialize_model_weights(
    model: torch.nn.Module,
    random_state: int = 42,
) -> None:
    """
    Reinitialize model parameters when the underlying module supports it.

    Args:
        model: Model whose parameters must be reinitialized.
        random_state: Random seed used for reproducibility.
    """
    torch.manual_seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    def reset_module(module: torch.nn.Module) -> None:
        """
        Reset parameters of a module when supported.
        """
        if hasattr(module, "reset_parameters"):
            module.reset_parameters()

    model.apply(reset_module)


@torch.no_grad()
def extract_features_from_loader(
    dataloader: DataLoader,
    pretrained_path: str,
    device: str,
    size: str = "base",
    modality: str = "both",
    image_resolution: int = 120,
    feature_type: str = "GAP",
    cache_dir: Optional[str] = None,
    cache_name: str = "features",
    cache_metadata: Optional[Dict] = None,
    force_recompute: bool = False,
    reinitialize_weights: bool = False,
    random_state: int = 42,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Extract features from a dataloader using PretrainedCROMA.

    Supported outputs:
        - modality="both", feature_type="GAP"        -> joint_GAP
        - modality="both", feature_type="encodings"  -> joint_encodings
        - modality="SAR", feature_type="GAP"         -> SAR_GAP
        - modality="SAR", feature_type="encodings"   -> SAR_encodings
        - modality="optical", feature_type="GAP"     -> optical_GAP
        - modality="optical", feature_type="encodings" -> optical_encodings

    If a cache path is configured and the cache exists, the function loads it
    directly instead of recomputing.

    Args:
        dataloader: Loader yielding (sar, optical, labels).
        pretrained_path: Path to converted CROMA weights.
        device: Device string.
        size: CROMA model size.
        modality: One of {"both", "SAR", "optical"}.
        image_resolution: Input image resolution.
        feature_type: One of {"GAP", "encodings"}.
        cache_dir: Optional cache directory.
        cache_name: Cache file prefix.
        cache_metadata: Optional metadata for deterministic cache naming.
        force_recompute: Whether to ignore existing cache.

    Returns:
        features: Tensor of shape (N, D) or (N, P, D).
        labels: Tensor of shape (N,).
    """
    if modality not in {"both", "SAR", "optical"}:
        raise ValueError(
            f"Unsupported modality '{modality}'. "
            "Expected one of {'both', 'SAR', 'optical'}."
        )

    if feature_type not in {"GAP", "encodings"}:
        raise ValueError(
            f"Unsupported feature_type '{feature_type}'. "
            "Expected one of {'GAP', 'encodings'}."
        )

    cache_path = None
    if cache_dir is not None:
        metadata = cache_metadata or {}
        full_metadata = {
            "pretrained_path": pretrained_path,
            "size": size,
            "modality": modality,
            "image_resolution": image_resolution,
            "feature_type": feature_type,
            "reinitialize_weights": reinitialize_weights,
            "random_state": random_state if reinitialize_weights else None,
        **metadata,
        }
        cache_path = build_cache_path(
            cache_dir=cache_dir,
            cache_name=cache_name,
            cache_metadata=full_metadata,
        )

        if cache_path.exists() and not force_recompute:
            payload = torch.load(cache_path, map_location="cpu")
            print(f"Loaded cached features from: {cache_path}")
            return payload["features"], payload["labels"]

    model = PretrainedCROMA(
        pretrained_path=pretrained_path,
        size=size,
        modality=modality,
        image_resolution=image_resolution,
    ).to(device)

    if reinitialize_weights:
        reinitialize_model_weights(
            model=model,
            random_state=random_state,
        )
    
    model.eval()

    if modality == "both":
        output_key = "joint_GAP" if feature_type == "GAP" else "joint_encodings"
    elif modality == "SAR":
        output_key = "SAR_GAP" if feature_type == "GAP" else "SAR_encodings"
    else:
        output_key = (
            "optical_GAP"
            if feature_type == "GAP"
            else "optical_encodings"
        )

    all_features = []
    all_labels = []

    for sar, optical, labels in dataloader:
        sar = sar.to(device, non_blocking=True)
        optical = optical.to(device, non_blocking=True)

        sar = croma_batch_normalize(sar)
        optical = croma_batch_normalize(optical)

        if modality == "both":
            outputs = model(SAR_images=sar, optical_images=optical)
        elif modality == "SAR":
            outputs = model(SAR_images=sar)
        else:
            outputs = model(optical_images=optical)

        features = outputs[output_key]

        all_features.append(features.detach().cpu())
        all_labels.append(labels.detach().cpu())

    features_tensor = torch.cat(all_features, dim=0)
    labels_tensor = torch.cat(all_labels, dim=0)

    if cache_path is not None:
        payload = {
            "features": features_tensor,
            "labels": labels_tensor,
            "metadata": full_metadata,
        }
        torch.save(payload, cache_path)
        print(f"Saved features to: {cache_path}")

    return features_tensor, labels_tensor


import os
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold


def cross_validate_features(
    features: torch.Tensor,
    labels: torch.Tensor,
    n_splits: int = 5,
    random_state: int = 42,
    max_iter: int = 1000,
) -> Dict[str, float]:
    """
    Perform stratified k-fold cross-validation on feature vectors.

    A logistic regression classifier is trained independently on each fold.

    Args:
        features: Tensor of shape (N, D).
        labels: Tensor of shape (N,).
        n_splits: Number of cross-validation folds.
        random_state: Random seed used to build the folds.
        max_iter: Maximum number of logistic regression iterations.

    Returns:
        Dictionary containing mean and standard deviation for accuracy
        and macro F1-score.
    """
    x = features.cpu().numpy()
    y = labels.cpu().numpy()

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    accuracy_scores = []
    f1_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(x, y)):
        x_train = x[train_idx]
        x_val = x[val_idx]

        y_train = y[train_idx]
        y_val = y[val_idx]

        '''
        clf = LogisticRegression(
            max_iter=max_iter,
            n_jobs=-1,
        )
        '''
        
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=max_iter,
                n_jobs=-1,
            ),
        )

        clf.fit(x_train, y_train)
        y_pred = clf.predict(x_val)

        accuracy = accuracy_score(y_val, y_pred)
        f1_macro = f1_score(
            y_val,
            y_pred,
            average="macro",
        )

        accuracy_scores.append(accuracy)
        f1_scores.append(f1_macro)

        print(
            f"Fold {fold + 1}: "
            f"accuracy={accuracy:.4f} "
            f"f1_macro={f1_macro:.4f}"
        )

    results = {
        "accuracy_mean": np.mean(accuracy_scores),
        "accuracy_std": np.std(accuracy_scores),
        "f1_macro_mean": np.mean(f1_scores),
        "f1_macro_std": np.std(f1_scores),
    }

    print("\n=== Cross-validation results ===")
    print(
        f"Accuracy: "
        f"mean={results['accuracy_mean']:.4f} "
        f"std={results['accuracy_std']:.4f}"
    )
    print(
        f"F1-macro: "
        f"mean={results['f1_macro_mean']:.4f} "
        f"std={results['f1_macro_std']:.4f}"
    )

    return results

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


test_h5 = "./LCZ42/testing.h5"

img_res = 120
batch_size = 64
num_workers = 4
device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = So2SatLCZ42Dataset(
    h5_path=test_h5,
    image_resolution=img_res,
    urban_only=True,
    selected_classes=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    remap_labels=True,
)

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)


# Full-model baseline
print("\n=== Full-model baseline ===")

full_model_path = "./checkpoints/croma_epoch_299_converted.pt"

full_features, full_labels = extract_features_from_loader(
    dataloader=loader,
    pretrained_path=full_model_path,
    device=device,
    size="base",
    modality="both",
    image_resolution=img_res,
    feature_type="GAP",
    cache_dir="./features",
    cache_name="full_model",
    cache_metadata={
        "split": "test",
        "dataset": test_h5,
        "urban_only": True,
        "selected_classes": list(range(10)),
        "remap_labels": True,
    },
    force_recompute=False,
    reinitialize_weights=False,
)

full_model_baseline = cross_validate_features(
    features=full_features,
    labels=full_labels,
    n_splits=5,
)


# Random baseline
print("\n=== Random baseline ===")

random_features, random_labels = extract_features_from_loader(
    dataloader=loader,
    pretrained_path=full_model_path,
    device=device,
    size="base",
    modality="both",
    image_resolution=img_res,
    feature_type="GAP",
    cache_dir="./features",
    cache_name="random_model",
    cache_metadata={
        "split": "test",
        "dataset": test_h5,
        "urban_only": True,
        "selected_classes": list(range(10)),
        "remap_labels": True,
    },
    force_recompute=False,
    reinitialize_weights=True,
    random_state=42,
)

random_baseline = cross_validate_features(
    features=random_features,
    labels=random_labels,
    n_splits=5,
)


print("\n=== Baselines ===")

print(
    "Full model - "
    f"F1={full_model_baseline['f1_macro_mean']:.4f} "
    f"+/- {full_model_baseline['f1_macro_std']:.4f}"
)

print(
    "Random model - "
    f"F1={random_baseline['f1_macro_mean']:.4f} "
    f"+/- {random_baseline['f1_macro_std']:.4f}"
)


# Selective training
print("\n=== Selective training ===")

selective_model_path = "./checkpoints/croma_30pct_one_step_converted.pt"

full_features, full_labels = extract_features_from_loader(
    dataloader=loader,
    pretrained_path=selective_model_path,
    device=device,
    size="base",
    modality="both",
    image_resolution=img_res,
    feature_type="GAP",
    cache_dir="./features",
    cache_name="test_split",
    cache_metadata={
        "split": "test",
        "dataset": test_h5,
        "urban_only": True,
        "selected_classes": list(range(10)),
        "remap_labels": True,
    },
    force_recompute=False,
    reinitialize_weights=False,
)

selective_training = cross_validate_features(
    features=full_features,
    labels=full_labels,
    n_splits=5,
)


# Scheduled selective training
print("\n=== Scheduled selective training ===")

scheduled_model_path = "./checkpoints/croma_30pct_stage_02_end_converted.pt"

features, labels = extract_features_from_loader(
    dataloader=loader,
    pretrained_path=scheduled_model_path,
    device=device,
    size="base",
    modality="both",
    image_resolution=img_res,
    feature_type="GAP",
    cache_dir="./features",
    cache_name="test_split",
    cache_metadata={
        "split": "test",
        "dataset": test_h5,
        "urban_only": True,
        "selected_classes": list(range(10)),
        "remap_labels": True,
    },
    force_recompute=False,
)

scheduled_training = cross_validate_features(
    features=features,
    labels=labels,
    n_splits=5,
)


# Selective finetuning
print("\n=== Selective finetuning ===")

finetuned_model_path = "./checkpoints/croma_30pct_finetuning_converted.pt"

features, labels = extract_features_from_loader(
    dataloader=loader,
    pretrained_path=finetuned_model_path,
    device=device,
    size="base",
    modality="both",
    image_resolution=img_res,
    feature_type="GAP",
    cache_dir="./features",
    cache_name="test_split",
    cache_metadata={
        "split": "test",
        "dataset": test_h5,
        "urban_only": True,
        "selected_classes": list(range(10)),
        "remap_labels": True,
    },
    force_recompute=False,
)

selective_finetuning = cross_validate_features(
    features=features,
    labels=labels,
    n_splits=5,
)


# Very selective finetuning
print("\n=== Very selective finetuning ===")

urban_finetuned_model_path = "./checkpoints/croma_3pct_finetuning_converted.pt"

features, labels = extract_features_from_loader(
    dataloader=loader,
    pretrained_path=urban_finetuned_model_path,
    device=device,
    size="base",
    modality="both",
    image_resolution=img_res,
    feature_type="GAP",
    cache_dir="./features",
    cache_name="test_split",
    cache_metadata={
        "split": "test",
        "dataset": test_h5,
        "urban_only": True,
        "selected_classes": list(range(10)),
        "remap_labels": True,
    },
    force_recompute=False,
)

very_selective_finetuning = cross_validate_features(
    features=features,
    labels=labels,
    n_splits=5,
)


print("\n=== Selective models ===")

print(
    "Selective model - "
    f"F1={selective_training['f1_macro_mean']:.4f} "
    f"+/- {selective_training['f1_macro_std']:.4f}"
)

print(
    "Selective scheduled model - "
    f"F1={scheduled_training['f1_macro_mean']:.4f} "
    f"+/- {scheduled_training['f1_macro_std']:.4f}"
)

print(
    "Selective finetuned model - "
    f"F1={selective_finetuning['f1_macro_mean']:.4f} "
    f"+/- {selective_finetuning['f1_macro_std']:.4f}"
)

print(
    "Very selective finetuned model - "
    f"F1={very_selective_finetuning['f1_macro_mean']:.4f} "
    f"+/- {very_selective_finetuning['f1_macro_std']:.4f}"
)


In [ ]:
from matplotlib import font_manager

font_path = "/usr/share/fonts/truetype/cmu/cmunss.ttf"
font_manager.fontManager.addfont(font_path)
font_prop = font_manager.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()

results = {
    "Full model": full_model_baseline,
    "Random model": random_baseline,
    "Selective training (30%)": selective_training,
    "Scheduled selective training (30%)": scheduled_training,
    "Selective finetuning (30%)": selective_finetuning,
    "Very selective finetuning (3%)": very_selective_finetuning,
}

model_names = list(results.keys())

f1_means = [
    results[name]["f1_macro_mean"]
    for name in model_names
]

f1_stds = [
    results[name]["f1_macro_std"]
    for name in model_names
]

fig, ax = plt.subplots(figsize=(5, 8))

bars = ax.bar(
    model_names,
    f1_means,
    yerr=f1_stds,
    capsize=5,
)

ax.set_ylabel("Macro F1-score")
ax.set_title("LCZ classification")
ax.set_ylim(0.7, 0.95)

ax.tick_params(
    axis="x",
    rotation=90,
)

for bar, mean in zip(bars, f1_means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{mean:.3f}",
        ha="center",
        va="bottom",
    )

fig.tight_layout()
plt.savefig(
    "lcz_model_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
import json

plot_results = {
    model_name: {
        "f1_mean": float(metrics["f1_macro_mean"]),
        "f1_std": float(metrics["f1_macro_std"]),
    }
    for model_name, metrics in results.items()
}

with open("lcz42_results.json", "w", encoding="utf-8") as file:
    json.dump(plot_results, file, indent=4)